In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
##### import project related moduls ####
current_file = Path.cwd() # cwd = path/*.ipynb - does not work in .py files.
print(f"current_file = {current_file}")
project_root = current_file.parent
print(f"project_root = {project_root}")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import utils.file_utils as fu   
from utils.terminal_styler import TerminalColours as tc
from spf2_converter import BGCorrector

%matplotlib QtAgg

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%reload_ext autoreload

### correction for forlder 2026_09_03_spectra_10Pa

#### get folder path

In [ ]:
## correction for forlder 2026_09_03_spectra_10Pa
# folder_path = None
folder_path = Path(r"C:\Andrei\DATA\VINETA_75\2026_09_03_spectra_10Pa\txt")
files = fu.list_files_in_folder(folder_path=folder_path)
print(files[0].parent)
for f in files:
    print(f.name)

#### set spectral and bg files

In [ ]:
spec_files_names = ["02sp1p3kW_10s.txt", "03sp1p3kW_10s.txt", 
              "04sp1p3kW_10s.txt", "05a2p0kW_10s.txt", 
              "06a2p0kW_10s.txt"]
bg_file_name = "abg_10s_16-52.txt"
corr = BGCorrector()
corr.bg_fpath = folder_path / bg_file_name
corr.data_dir = folder_path

print(corr.bg_fpath)


#### correction

In [ ]:
for name in spec_files_names:
    fpath = folder_path / name
    corr.spec_fpath = fpath
    corr.correct_file(tx_A=10, tx_B=10)
    

#### check bg files

In [ ]:
dfs = []
fpath = folder_path / "bg_10s_15-37.txt"
df1537 = corr.read_file(fpath=fpath)
dfs.append(df1537)
fpath = folder_path / "bg_10s_15-48.txt"
df1548 = corr.read_file(fpath=fpath)
dfs.append(df1548)
fpath = folder_path / "bg_10s_16-14.txt"
df1614 = corr.read_file(fpath=fpath)
dfs.append(df1614)
fpath = folder_path / "abg_10s_16-52.txt"
dfa = corr.read_file(fpath=fpath)
dfs.append(dfa)

len(dfs)
dfa.head(10)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 5), dpi=100)
for df in dfs:
    ax1.plot(df["wavelength1"], df["intensity1"],  alpha=0.8)
    ax2.plot(df["wavelength2"], df["intensity2"],  alpha=0.8)

### correction for forlder 2026_09_04_spectra_10Pa

#### get folder path

In [ ]:
folder_path = Path(r"C:\Andrei\DATA\VINETA_75\2026_09_04_spectra_10Pa\txt")
files = fu.list_files_in_folder(folder_path=folder_path)
print(files[0].parent)
for f in files:
    print(f.name)

In [ ]:
%reload_ext autoreload

#### set spectral and bg files

In [ ]:
## seporate bg and other files
files = fu.list_files_in_folder(folder_path=folder_path)
bg_files = []
spec_files = []
for file in files:
    if file.name.endswith('_bg.txt'):
        bg_files.append(file)
    else:
        spec_files.append(file)

for f in bg_files:
    print(f.name)


In [ ]:
## get spectral files with bg piar
working_files = []
for f in spec_files:
    f_bg = f.parent / f"{f.stem}_bg.txt"
    if f_bg in bg_files:
        print(f"{tc.GREEN} {f.stem} {tc.RESET}")
        working_files.append(f)
    else:
        print(f"{tc.RED}{f.stem} {tc.RESET}")

In [ ]:
%reload_ext autoreload

In [ ]:
corr = BGCorrector()
corr.data_dir = working_files[0].parent
for f in working_files:
    sp_name = f.name.split('_')
    corr.fname_parts = sp_name
    corr.spec_fpath = f
    corr.bg_fpath = f.parent / f"{f.stem}_bg.txt"

    # print(corr.fname_parts[-1])
    if corr.fname_parts[-1].endswith("s.txt"):
        tx = int(corr.fname_parts[-1].split('.')[0][0])
    else: 
        tx = 10
    print(f"{tx=}")
    corr.correct_file(tx_A=tx, tx_B=tx)


    print()
 